# Wrong-way detection on a Colab GPU

Runs `pipeline.py` (RF-DETR + ByteTrack + `WrongWayDetector`) against
traffic video.

The code executes on a **remote Google machine**, which cannot see your
local disk — so the cells below fetch the code from GitHub and the video
from either the `supervision` sample set, Drive, or an upload.

**Pick a GPU runtime before running:** Runtime → Change runtime type →
T4 GPU. The first cell says plainly whether you got one. On CPU
everything still works, roughly an order of magnitude slower.

**Read the class-id table in the run cell's output before you read any
alert.** COCO has two numbering schemes that disagree about every
vehicle, and picking the wrong one means tracking bicycles and trains
while every bus and truck is silently discarded — which is exactly what
happened here for two runs. The table now shows both interpretations
where the model will not name its own classes; resolve it by looking at
what is actually on the road in the footage.

**Two rules that have each cost an hour:**

- After every push, re-run the git cell. A running kernel keeps executing
  the code already in memory, so a fixed bug reproduces identically.
- If anything behaves as though the old code is still running, restart
  the runtime and run from the top. Three minutes of re-running beats
  twenty minutes chasing a bug that was already fixed.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Switch the runtime to a GPU type, then rerun this cell.")

## 1. Install

Only two packages. The runtime already ships `torch` (CUDA build),
`opencv` and `numpy`; installing `requirements.txt` wholesale can replace
the CUDA torch with a CPU one and silently cost you the GPU.
`supervision` arrives as a dependency of `rfdetr`.

In [ ]:
!pip install -q rfdetr trackers

## 2. Get the code

Clones on the first run of a session, pulls on every run after that.
**Re-run this cell after every push** — it is what carries your local
edits across to the machine that actually executes them.

It also clears our modules from Python's import cache. Pulling changes
the files on disk, but a running kernel keeps executing the copies
already in memory, so a fixed bug reproduces identically and the fix
looks like it failed. That is the most common reason "I already fixed
that" turns out to be false in any Jupyter session.

If anything still behaves as though the old code is running, restart the
runtime and run from the top. That always works, and three minutes of
re-running beats twenty minutes chasing a bug that was already fixed.

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Arielevi15/Crime_Traffic_Dedector.git"
REPO_DIR = "/content/Crime_Traffic_Dedector"

if not os.path.isdir(REPO_DIR):
    !git clone --quiet {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

# Drop our modules from Python's import cache. Without this, `git pull`
# updates the files on disk while the kernel keeps executing the copies
# already in memory -- a fixed bug reproduces identically and the fix
# looks like it failed. The next import below re-reads from disk.
#
# (IPython's %autoreload would also do this, but it imports `imp`, which
# Python 3.12 removed, so it cannot load on this runtime at all.)
for _name in ("pipeline", "wrong_way_detector", "replay"):
    sys.modules.pop(_name, None)

!ls

## 3. Get the video

Two options. Run **3a** for the very first run, and **3b** once you have
real footage -- they answer different questions.

### 3a. Sample footage (start here)

One line, no upload, no Drive. `supervision` ships it, and it is already
installed as a dependency of `rfdetr`.

Be clear about what this does and does not prove. It is **elevated
highway footage, not a forward-facing dashcam**, so it does not match the
scope assumption at the top of `CLAUDE.md`. Good for: confirming the
class ids, that RF-DETR loads, that ByteTrack holds ids across frames,
and that the chain runs end to end. Not good for: tuning any threshold in
`DetectorConfig`, or judging the wrong-way logic in the domain we
actually care about.

In [ ]:
from supervision.assets import VideoAssets, download_assets

VIDEO = download_assets(VideoAssets.VEHICLES)
print("Video ready:", VIDEO)

### 3b. Your own dashcam footage

Skip this on the first run. Once you have real clips, put them in a Drive
folder once and every future session sees them without another upload.
Mounting opens an auth prompt the first time. Running this cell
overwrites `VIDEO` from 3a.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Point this at your own clip. Not sure of the exact path? Find it:
#   !find /content/drive/MyDrive -iname "*.mp4" | head -20
# Note that Drive's "Shared with me" is a view, not a folder -- it is
# never mounted. Add a shortcut to My Drive, or copy the file there.
candidate = "/content/drive/MyDrive/dashcam/sample1.mp4"

# Check first, assign second. Assigning before the check would let a
# wrong path here silently replace a working VIDEO set by cell 3a, and
# the failure would then surface much later, in the run cell.
assert os.path.isfile(candidate), (
    "Not found: {0}\nRun the find command above to check the name.".format(candidate)
)
VIDEO = candidate
print("Video ready:", VIDEO)

## 4. Run

`limit_frames=300` keeps the first run short: long enough to produce the
class-id table and show whether tracking holds, short enough that a
misconfiguration costs seconds rather than an hour. Drop it once the
class ids are confirmed.

In [ ]:
from pipeline import run

# One name, defined once and reused by the download cell. Hardcoding the
# filename in two places is how you end up downloading a stale dump from
# an earlier run and debugging output the code never produced.
TRACKS = "tracks.jsonl"

alerts = run(
    video=VIDEO,
    output="check.mp4",
    limit_frames=300,
    # Records what the violation modules actually consume. See section 7 --
    # this is what makes logic debugging fast.
    dump_tracks=TRACKS,
)
alerts

## 5. Watch the annotated result

Green box = tracked vehicle, red = alerted, orange dot = the
road-contact point the detector actually reasons about. If those dots are
not landing on the road beneath each vehicle, fix that before tuning any
threshold -- everything downstream depends on that point being right.

OpenCV writes `mp4v`, which the notebook player will not decode, so
re-encode to H.264 first. The video is inlined as base64, which is fine
for a few hundred frames; for a full clip, download it instead.

In [ ]:
from base64 import b64encode

from IPython.display import HTML

!ffmpeg -loglevel error -i check.mp4 -vcodec libx264 -y check_h264.mp4

payload = b64encode(open("check_h264.mp4", "rb").read()).decode()
HTML('<video width=720 controls><source src="data:video/mp4;base64,{0}">'.format(payload))

In [ ]:
# Longer clips: copy the result back to Drive instead of inlining it.
!cp check_h264.mp4 /content/drive/MyDrive/dashcam/

## 6. Tuning

Once the class ids are confirmed, the next open task is tuning
`DetectorConfig` against real footage. Pass one in explicitly rather than
editing the module, so the tested defaults stay intact:

```python
from wrong_way_detector import DetectorConfig

alerts = run(
    video=VIDEO,
    output="check.mp4",
    config=DetectorConfig(opposite_cos_threshold=-0.6),
)
```

Per `CLAUDE.md` principle 3, tune toward silence. A false positive
accuses an innocent driver; a false negative merely misses one.

## 7. Take the track data home — stop iterating through the GPU

The run cell wrote `tracks.jsonl`: per frame, every track's id and
road-contact point. That is the entire input surface a violation module
has (`CLAUDE.md` principle 5), so **everything downstream of perception
can be reproduced from it exactly** — with no GPU, no model, no video and
no third-party packages.

This matters because of arithmetic. Iterating on the logic through this
notebook costs minutes per attempt: edit, push, pull, reload the model,
decode video. Replaying the same run locally costs under a second. When
the thing being debugged is the logic rather than the perception — which
is nearly always — there is no reason to pay the GPU cost again.

Download the file, drop it in the project folder, and work locally:

```
python replay.py tracks.jsonl
python replay.py tracks.jsonl --track 16 --verbose
python replay.py tracks.jsonl --zone-size 240 --opposite-cos-threshold -0.6
```

`--track N --verbose` prints that one vehicle's per-frame decision trail —
heading, zone, whether the zone was trusted, cosine, streak. It is the
fastest way to see why a specific alert fired. The threshold flags let a
parameter sweep be a shell loop instead of a code edit.

In [ ]:
import glob
import os

from google.colab import files

# Prefer the name the run cell used; fall back to whatever dump is newest.
# Never assume a file is there -- an empty runtime and a stale file from an
# earlier run look identical from here, and the second one is worse,
# because you end up analysing output the current code never produced.
target = globals().get("TRACKS")
if not target or not os.path.isfile(target):
    dumps = sorted(glob.glob("*.jsonl"), key=os.path.getmtime, reverse=True)
    print("dumps present:", dumps or "none")
    target = dumps[0] if dumps else None

if target is None:
    print("No dump here yet. Run the pipeline cell first.")
else:
    print("downloading {0} ({1} KB, written {2})".format(
        target,
        os.path.getsize(target) // 1024,
        __import__("time").ctime(os.path.getmtime(target)),
    ))
    files.download(target)

## 8. Build a corpus without downloading anything by hand

Everything above processes one clip that somebody put there. That does not
scale to the twenty-odd clips threshold tuning needs, and nobody should be
downloading videos to a laptop to re-upload them.

`corpus.py` names a source and the clips arrive:

```python
from corpus import build_dumps, fetch

clips = fetch("hf:smart-dashcam/motorcycle-accident-driving-datasets", limit=10)
build_dumps(clips, out_dir="fixtures", limit_frames=600)
```

Sources are `url:`, `hf:`, `kaggle:`, or a path. Hugging Face public
repositories need no credentials; Kaggle needs an API token once per
machine. Loose video files are preferred, and archives are searched
otherwise — most driving datasets ship WebDataset tarballs rather than
loose clips, so a fetcher that only understands loose files finds nothing
in them.

**What comes back is dumps, not video.** A clip is ~20 MB and needs a GPU;
its dump is ~200 KB, needs nothing, and is the whole of what a violation
module ever sees. So video is fetched once on a machine that does not care
— this one — and only the dumps travel home, into `fixtures/`, where they
replay for free forever and reach your partner through `git pull`.

**None of this needs labelling.** Ordinary driving footage contains no
wrong-way driving to any useful approximation, so every alert raised on it
is one that should not have been raised — a false-positive rate measured
straight from unlabelled video. Positives come from turning real
trajectories around instead, which is what `inject.py` is for.

In [ ]:
!pip install -q huggingface_hub kagglehub

import sys

for _m in ("corpus", "pipeline", "wrong_way_detector"):
    sys.modules.pop(_m, None)
from corpus import build_dumps, fetch

# Pick a source. Start small: ten clips is enough to see whether the corpus
# route works before spending an hour on a hundred.
SOURCE = "hf:smart-dashcam/motorcycle-accident-driving-datasets"

clips = fetch(SOURCE, limit=10)
print("\n{0} clip(s) fetched\n".format(len(clips)))

# limit_frames keeps a first corpus run to minutes rather than an hour.
# Drop it once the route is proven.
build_dumps(clips, out_dir="fixtures", limit_frames=600)

In [ ]:
import glob
import os
import shutil

from google.colab import files

# Bring the whole corpus home in one archive. Dumps are small enough that
# a hundred of them still fit comfortably in a repository.
dumps = sorted(glob.glob("fixtures/*.jsonl"))
total = sum(os.path.getsize(path) for path in dumps)
print("{0} dump(s), {1} KB total".format(len(dumps), total // 1024))

if dumps:
    shutil.make_archive("corpus", "zip", "fixtures")
    print("corpus.zip:", os.path.getsize("corpus.zip") // 1024, "KB")
    files.download("corpus.zip")